# Apriori

In [1]:
!pip install mlxtend

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   -------------------------------------- - 1.3/1.4 MB 22.3 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 14.0 MB/s  0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ---------------------- ----------------- 6.8/12.3 MB 32.3 MB/s eta 0:00:01
   ---------------------------------------  12.1/12.3 MB 32.8 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 26.6 MB/s  0:00:00

  Attempting uninstall: numpy

    Found existing installation: numpy 2.0.2

   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
    Uninstalling numpy-2.0.2:
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ------------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.63.1 requires numpy<2.4,>=1.22, but you have numpy 2.4.3 which is incompatible.
tensorflow-intel 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.4.3 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Import required libraries
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

In [4]:
# Each row represents a shopping basket or transaction ID
transactions = [
    ['Milk', 'Bread', 'Butter'], # Customer bought: Milk + Bread + Butter
    ['Bread', 'Butter'],
    ['Milk', 'Bread'],
    ['Milk', 'Butter'],
    ['Bread', 'Butter'],
    ['Milk', 'Bread', 'Butter', 'Eggs'],
    ['Milk', 'Eggs'],
    ['Bread', 'Eggs'],
    ['Milk', 'Bread', 'Butter'],
    ['Butter', 'Eggs']
]

# print("Sample Transactions:")
# for t in transactions:
#     print(t)

In [5]:
# Convert transactions into one-hot encoded dataframe
# Apriori requires a **boolean table**.

te = TransactionEncoder()

te_array = te.fit(transactions).transform(transactions)

df = pd.DataFrame(te_array, columns=te.columns_)

print(df)

   Bread  Butter   Eggs   Milk
0   True    True  False   True
1   True    True  False  False
2   True   False  False   True
3  False    True  False   True
4   True    True  False  False
5   True    True   True   True
6  False   False   True   True
7   True   False   True  False
8   True    True  False   True
9  False    True   True  False


In [6]:
# Find frequent itemsets with minimum support

frequent_itemsets = apriori(df, min_support=0.3, use_colnames=True)

print("Frequent Itemsets:")
print(frequent_itemsets)

Frequent Itemsets:
   support               itemsets
0      0.7                (Bread)
1      0.7               (Butter)
2      0.4                 (Eggs)
3      0.6                 (Milk)
4      0.5        (Bread, Butter)
5      0.4          (Bread, Milk)
6      0.4         (Butter, Milk)
7      0.3  (Bread, Butter, Milk)


**Support**


Support(A) = Transactions containing A / Total transactions


Example:

Milk appears in **6 of 10 transactions**

Support (Milk) = **0.6**


In [7]:
# Now we generate rules from frequent itemsets.

rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.6
)

print("Association Rules:")
print(rules[['antecedents','consequents','support','confidence','lift']])

Association Rules:
       antecedents consequents  support  confidence      lift
0          (Bread)    (Butter)      0.5    0.714286  1.020408
1         (Butter)     (Bread)      0.5    0.714286  1.020408
2           (Milk)     (Bread)      0.4    0.666667  0.952381
3           (Milk)    (Butter)      0.4    0.666667  0.952381
4  (Bread, Butter)      (Milk)      0.3    0.600000  1.000000
5    (Bread, Milk)    (Butter)      0.3    0.750000  1.071429
6   (Butter, Milk)     (Bread)      0.3    0.750000  1.071429


### Interpretation:

Customers who buy Bread are 71% likely to buy Butter.

In [8]:
# Sort Rules by Strength
rules_sorted = rules.sort_values(by='lift', ascending=False)

print(rules_sorted[['antecedents','consequents','support','confidence','lift']])

       antecedents consequents  support  confidence      lift
6   (Butter, Milk)     (Bread)      0.3    0.750000  1.071429
5    (Bread, Milk)    (Butter)      0.3    0.750000  1.071429
0          (Bread)    (Butter)      0.5    0.714286  1.020408
1         (Butter)     (Bread)      0.5    0.714286  1.020408
4  (Bread, Butter)      (Milk)      0.3    0.600000  1.000000
2           (Milk)     (Bread)      0.4    0.666667  0.952381
3           (Milk)    (Butter)      0.4    0.666667  0.952381


# Example:
### People who buy (Butter, Milk) are 75% likely to buy Bread